In [1]:
from tinkerbell.sampling.server import SGLangServerConfig, ModalDeployConfig, wait_for_server
from tinkerbell.sampling.client import SamplingClient

In [14]:
small_model = "Qwen/Qwen3-0.6B"
big_model = "Qwen/Qwen3-30B-A3B-Instruct-2507"

In [27]:
modal_config = ModalDeployConfig(
    gpu="H100",
    num_gpus=4,
)
config = SGLangServerConfig(
    server_url="https://0.0.0.0:8000",
    model_name=big_model,
)
client = SamplingClient.deploy(
    server_config=config,
    deploy_config=modal_config
)

/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/tinkerbell/sampling/server.py:330: DeprecationError: 2025-02-24: We have renamed several parameters related to autoscaling. Please update your code to use the following new names:

- container_idle_timeout -> scaledown_window

See https://modal.com/docs/guide/modal-1-0-migration for more details.
  @app.cls(


⠼ Creating objects...
├── 🔨 Created mount PythonPackage:tinkerbell
├── 🔨 Created function SGLangService.*.
└── 🔨 Created web endpoint for SGLangService.serve => 
    https://jesterlabs--sglang-model.modal.run
✓ Created objects.
├── 🔨 Created mount PythonPackage:tinkerbell
├── 🔨 Created function SGLangService.*.
└── 🔨 Created web endpoint for SGLangService.serve => 
    https://jesterlabs--sglang-model.modal.run
✓ App deployed in 1.814s! 🎉

View Deployment: 
https://modal.com/apps/jesterlabs/main/deployed/tinkerbell-sglang-inference-sgla
ng-model


In [28]:
client.server_url

'https://jesterlabs--sglang-model.modal.run'

In [29]:
wait_for_server(client.server_url)

True

In [30]:
status = client.health()

In [33]:
print(status)

{'status': 'healthy', 'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507'}


In [34]:
from pydantic import BaseModel
class Character(BaseModel):
    name: str
    description: str
    personality: str
    backstory: str
    appearance: str
    dialogue: str

schema = Character.model_json_schema()
print(schema)

{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'description': {'title': 'Description', 'type': 'string'}, 'personality': {'title': 'Personality', 'type': 'string'}, 'backstory': {'title': 'Backstory', 'type': 'string'}, 'appearance': {'title': 'Appearance', 'type': 'string'}, 'dialogue': {'title': 'Dialogue', 'type': 'string'}}, 'required': ['name', 'description', 'personality', 'backstory', 'appearance', 'dialogue'], 'title': 'Character', 'type': 'object'}


In [24]:
import json

prompts = ["Please generate a fantasy character."]
sampling_params = {
    "max_new_tokens": 1024,
    "temperature": 1.0,
    "json_schema": json.dumps(schema),
}
generate_kwargs = {}


In [25]:
generate_response = client.generate(prompts, sampling_params, generate_kwargs)

In [26]:
generate_response

[{'text': '{\n  "name": "Draven Shadowfang",\n  "description": "An ashen-skinned elven sorcerer whose eyes glow faintly in the dark. He possesses cursed magic that twists nature into macabre forms—twisted trees, blood-red vines, and skeletal beasts that obey his every command. He speaks in hushed whispers, and even his breath seems to carry the scent of decay.",\n  "personality": "melancholic, brooding, obsessive, independant, has a hidden compassion",\n  "backstory": "Draven was once a gifted young elven scholar of ancient arcana, revered for his wisdom. When he discovered a forbidden spell to resurrect his deceased lover, he inadvertently summoned a primordial entity from the Darkveil—a place beyond known realms. The entity cursed him, warping his magic and turning his body into a living vessel of decay. Now, he wanders the cursed lands, consumed by guilt and grief, yet instinctively driven to protect innocent lives from the very horrors he creates. He is known as the Withering Sage—

In [ ]:
import modal
deployed_app = modal.App.lookup("tinkerbell-sglang-inference-sglang-model", create_if_missing=False)

In [ ]:
deployed_app.registered_entrypoints

In [ ]:
import modal

# Get workspace from config
workspace = modal.config._profile
print(workspace)

In [ ]:
import os
workspace = os.environ.get("MODAL_WORKSPACE", "default")
print(workspace)

In [ ]:
client.health()

In [ ]:
import modal

In [ ]:
deployed_app = modal.App.lookup("tinkerbell-sglang-inference-sglang-model", create_if_missing=False)


In [ ]:
from pydantic import BaseModel
from typing import Dict, Any
from pydantic import Field


class ServerConfig(BaseModel):
    server_url: str
    model_name: str
    tokenizer: str | None = None


class SGLangServerConfig(ServerConfig):
    engine_kwargs: Dict[str, Any] = Field(default_factory=dict)



In [ ]:
def get_host_and_port(server_url: str) -> tuple[str, int | None]:
    """
    Parse server URL to extract host and port.

    Args:
        server_url: URL in format "http://host:port", "host:port", "http://host", or "host"

    Returns:
        Tuple of (host, port) where port is None if not specified
    """
    from urllib.parse import urlparse

    # Add scheme if not present to help urlparse
    if not server_url.startswith(('http://', 'https://')):
        server_url = 'http://' + server_url

    parsed = urlparse(server_url)
    host = parsed.hostname or parsed.netloc.split(':')[0]
    port = parsed.port

    return host, port

In [ ]:
get_host_and_port("https://0.0.0.0:8000")

In [ ]:
import httpx
import asyncio

In [ ]:
with httpx.Client(timeout=10) as client:
    x = client.get("https://example.com")

In [ ]:
type(x)

In [ ]:
class A:
    def __init__(self, a: int):
        self.a = a

    def __str__(self):
        return f"A({self.a})"

class B(A):

    def test(self):
        print("Value of a is", self.a)

In [ ]:
b = B(10)

In [ ]:
b.test()

In [ ]:
from fastapi import FastAPI
import asyncio

app = FastAPI()

@app.get("/test_async")
async def test_async(x: str):
    """
    Test async endpoint that waits 5 seconds with logging
    """
    for i in range(5):
        await asyncio.sleep(1)
        print(f"Waiting {i} seconds for X value {x}")
    
    return {"message": f"Completed waiting for {x}", "value": x}

# Optional: Add a root endpoint
@app.get("/")
async def root():
    return {"message": "FastAPI async test server"}


In [ ]:
# Cell 2: Run the server in a background thread
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://localhost:8000")

In [ ]:
import httpx
import asyncio

class TestClient:
    def __init__(self, timeout: float = 10.0):
        self.timeout = timeout

    async def test_concurrent_async(self, x: str):
        async with httpx.AsyncClient(timeout=self.timeout) as client:
        # tasks = [
        #     client.get("http://localhost:8000/test_async?x=first"),
        #     client.get("http://localhost:8000/test_async?x=second"),
        #     client.get("http://localhost:8000/test_async?x=third"),
        # ]
        # responses = await asyncio.gather(*tasks)
        # for r in responses:
        #     print(r.json())
            output = await client.get(f"http://localhost:8000/test_async?x={x}")
            return output.json()

    def test_concurrent_sync(self, x: str):
        with httpx.Client(timeout=self.timeout) as client:
            return client.get(f"http://localhost:8000/test_async?x={x}").json()

client = TestClient()
future1 = client.test_concurrent_async("test-first")
future2 = client.test_concurrent_async("test-second")
future3 = client.test_concurrent_async("test-third")
responses = await asyncio.gather(future1, future2, future3)


In [ ]:
print(responses)

In [ ]:
output = client.test_concurrent_sync("first")

In [ ]:
output

In [ ]:
strings  = ["first", "second", "third"]
future1 = client.test_concurrent("first")
future2 = client.test_concurrent("second")
future3 = client.test_concurrent("third")
# await asyncio.gather(*futures)


In [ ]:
future1

In [ ]:
responses = await asyncio.gather(*[future1, future2, future3])

In [ ]:
responses[0].get()

In [ ]:
futures[0].result()

In [ ]:
async def wait_test(x):
    import asyncio
    for i in range(5):
        await asyncio.sleep(1)  # ✅ This yields control to the event loop!
        print(f"Waiting {i} seconds for X value {x}")
    return x

future = asyncio.ensure_future(wait_test(20))
future2 = asyncio.ensure_future(wait_test(30))

In [ ]:
future = asyncio.ensure_future(wait_test(20))
future2 = asyncio.ensure_future(wait_test(30))

In [ ]:
out = future.result() * future2.result()

In [ ]:
future.

In [ ]:
print(out)

In [ ]:
from typing import Any, Callable
from pydantic import BaseModel
from dataclasses import asdict

class TrainingClientConfig(BaseModel):
    model_name: str | None = "test"
    model_creation_function: Callable | None = None
    model_kwargs: dict | None = None
    lora_config: dict | None = None
    model_checkpoint_path: str | None = None


In [ ]:
a = TrainingClientConfig()

In [ ]:
a.model_dump()

In [ ]:
!pip install sglang

In [ ]:
!pip install pybase64

In [ ]:
from sglang import Engine

In [ ]:
import sglang

In [ ]:
print(sglang.__version__)

In [ ]:
from fastapi import FastAPI, Request, Response
from fastapi.responses import Response as FastAPIResponse
import nest_asyncio
nest_asyncio.apply()

In [ ]:
engine = Engine(model_name="Qwen/Qwen3-0.6B")

In [ ]:
!pip install triton

In [ ]:
"""Utility functions for efficient tensor serialization/deserialization."""

import io
from typing import Any
import torch
import uvicorn

def serialize_tensor(obj: Any) -> bytes:
    """Serialize a tensor or nested structure of tensors to bytes.
    Args:
        obj: A torch tensor, list, dict, or nested structure containing tensors
    Returns:
        bytes: Serialized representation
    """
    buffer = io.BytesIO()
    torch.save(obj, buffer, _use_new_zipfile_serialization=False)
    buffer.seek(0)
    return buffer.read()


def deserialize_tensor(data: bytes) -> Any:
    """Deserialize bytes back to tensor or nested structure.

    Args:
        data: Serialized bytes from serialize_tensor
    Returns:
        The deserialized tensor or nested structure
    """
    buffer = io.BytesIO(data)
    buffer.seek(0)
    return torch.load(buffer, map_location='cpu')


def serialize_payload(data: list[Any], loss_fn: Any = None, **kwargs) -> bytes:
    """Serialize a complete payload including data and additional parameters.

    Args:
        data: List of data points (can contain tensors)
        loss_fn: Optional loss function or serializable representation
        **kwargs: Additional parameters to serialize

    Returns:
        bytes: Serialized payload
    """
    payload = {
        'data': data,
        'loss_fn': loss_fn,
        **kwargs
    }
    return serialize_tensor(payload)


def deserialize_payload(data: bytes) -> dict:
    """Deserialize a complete payload.

    Args:
        data: Serialized bytes

    Returns:
        dict: Deserialized payload with 'data', 'loss_fn', and other fields
    """
    return deserialize_tensor(data)


class TorchServer:
    def __init__(
        self,
        host: str = "0.0.0.0",
        port: int = 8000
    ):
        self.app = FastAPI()
        self.host = host
        self.port = port
        self.setup_routes()

    def setup_routes(self):
        @self.app.post("/forward")
        async def forward_endpoint(request: Request):
            """Forward pass endpoint."""
            body = await request.body()
            payload = deserialize_payload(body)
            data = payload['data']
            out = data*20.0
            serialized_result = serialize_tensor(out)
            return FastAPIResponse(content=serialized_result, media_type="application/octet-stream")

        @self.app.get("/health")
        async def health_check():
            """Health check endpoint."""
            return {"status": "healthy"}

    def deploy(self, config: dict | None = None) -> Any:
        """Deploy the server.
        
        Args:
            config: Optional configuration dictionary (can override host/port)
            
        Returns:
            Server configuration
        """
        if config:
            host = config.get('host', self.host)
            port = config.get('port', self.port)
        else:
            host = self.host
            port = self.port
        
        print(f"Starting Torch Training Server on {host}:{port}")
        uvicorn.run(self.app, host=host, port=port)
        return {"host": host, "port": port}



In [ ]:
server = TorchServer()
server.deploy()